## **Installing and Importing Dependencies**

In [6]:
!pip install -q gradio openai pypdf python-docx fpdf2

In [7]:
import os
import re
import base64
import tempfile
import mimetypes
import subprocess
from datetime import datetime
import requests
import gradio as gr
from openai import OpenAI
from pypdf import PdfReader
import docx
from fpdf import FPDF

# **1. Setup DeepSeek API Key & Client**

In [8]:

try:
    from google.colab import userdata
    DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
except Exception:
    DEEPSEEK_API_KEY = "your-deepseek-api-key-here"  # Or paste your key directly here

DEEPSEEK_BASE_URL = "https://api.deepseek.com"

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

## Warm initial greeting from Kumar

In [9]:

INITIAL_GREETING = (
    "Hey there! 👋 I am **Kumar**, your personal AI assistant. "
    "It is wonderful to have you here! You can type, click the microphone to talk to me, "
    "search the live web, or attach images and documents. How can I make your day easier today?"
)

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".gif"}
AUDIO_EXTENSIONS = {".wav", ".mp3", ".ogg", ".opus", ".webm", ".m4a", ".flac"}

## Model choices mapped to (`model_id`, `enable_thinking`)

In [10]:

MODEL_OPTIONS = {
    "DeepSeek-V4.1-Flash (Fast + Vision + Web)": ("deepseek-flash", False),
    "DeepSeek-V4.1-Flash Thinking (Deep Search + Vision)": ("deepseek-flash", True),
    "DeepSeek-V4-Pro (Deep Text & Code)": ("deepseek-v4-pro", True),
}

## DeepSeek TTS voices ("tide" is male, "stella" is female)

In [11]:

VOICE_OPTIONS = {
    "Tide (Male — Kumar's Voice)": "tide",
    "Stella (Female)": "stella",
}

# **2. Conversation Export Helpers (Markdown & PDF)**

## Extractig Clean Texts from the Chat

In [12]:
def extract_clean_turns(history):
    """Normalizes Gradio 6.0 chat history into clean (speaker, text, attachments) tuples."""

    turns = []

    for msg in history or []:
        role = msg.get("role", "assistant")
        speaker = "You" if role == "user" else "Kumar"
        raw_content = msg.get("content", "")

        text_parts = []
        attachments = []

        if isinstance(raw_content, str):
            text_parts.append(raw_content)

        elif isinstance(raw_content, list):
            for block in raw_content:
                if isinstance(block, dict):

                    if block.get("type") == "text":
                        text_parts.append(block.get("text", ""))

                    elif block.get("type") == "file" and "file" in block:
                        fpath = block["file"].get("path", "")
                        fname = os.path.basename(fpath)

                        # Skip temporary TTS audio clips in assistant bubbles
                        if not (role == "assistant" and fname.startswith("tmp") and fname.endswith(".wav")):
                            attachments.append(fname)

        combined_text = "\n".join(text_parts).strip()
        # Strip UI routing banners and HTML <details> wrappers for clean export
        combined_text = re.sub(r"^\*\(Switched to.*?\)\*\s*", "", combined_text)
        combined_text = re.sub(
            r"<details.*?>\s*<summary>.*?</summary>\s*(.*?)\s*</details>\s*",
            r"[Thought Process]\n\1\n\n",
            combined_text,
            flags=re.DOTALL
        ).strip()

        if combined_text or attachments:
            turns.append((speaker, combined_text, attachments))

    return turns

## Export to Markdown

In [13]:
def export_to_markdown(history):
    turns = extract_clean_turns(history)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    file_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = os.path.join(tempfile.gettempdir(), f"Kumar_Conversation_{file_stamp}.md")

    lines = [
        "# 🤖 Kumar — Conversation Transcript",
        f"*Exported on {timestamp}*",
        "",
        "---",
        ""
    ]

    for speaker, text, attachments in turns:
        icon = "👤" if speaker == "You" else "🤖"
        lines.append(f"### {icon} {speaker}")

        if attachments:
            lines.append(f"*Attachments: {', '.join(attachments)}*")
            lines.append("")
        if text:
            lines.append(text)

        lines.extend(["", "---", ""])

    lines.append("*This superintelligent chatbot is built by Subhadeep Dey*")

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    return out_path

## Export to PDF

In [14]:
def export_to_pdf(history):
    turns = extract_clean_turns(history)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    file_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = os.path.join(tempfile.gettempdir(), f"Kumar_Conversation_{file_stamp}.pdf")

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    # Load Unicode TrueType font (available by default in Google Colab) with Latin-1 fallback
    dejavu_regular = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
    dejavu_bold = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
    use_unicode = os.path.exists(dejavu_regular) and os.path.exists(dejavu_bold)

    if use_unicode:
        pdf.add_font("DejaVu", "", dejavu_regular)
        pdf.add_font("DejaVu", "B", dejavu_bold)
        font_family = "DejaVu"
    else:
        font_family = "Helvetica"

    def safe_text(val):
        # Strip emojis that standard fonts lack while keeping full text & punctuation
        cleaned = re.sub(r"[\U00010000-\U0010ffff]", "", val)
        if not use_unicode:
            cleaned = cleaned.encode("latin-1", "ignore").decode("latin-1")

        return cleaned

    # Title Banner
    pdf.set_font(font_family, "B", 16)
    pdf.set_text_color(25, 45, 85)
    pdf.cell(0, 10, safe_text("Kumar — AI Conversation Transcript"), new_x="LMARGIN", new_y="NEXT")

    pdf.set_font(font_family, "", 9)
    pdf.set_text_color(110, 110, 110)
    pdf.cell(0, 6, safe_text(f"Exported on {timestamp}  |  Built by Subhadeep Dey"), new_x="LMARGIN", new_y="NEXT")
    pdf.ln(4)

    # Conversation Turns
    for speaker, text, attachments in turns:
        pdf.set_font(font_family, "B", 11)

        if speaker == "You":
            pdf.set_text_color(20, 90, 160)
        else:
            pdf.set_text_color(20, 120, 80)

        pdf.cell(0, 7, safe_text(speaker), new_x="LMARGIN", new_y="NEXT")

        if attachments:
            pdf.set_font(font_family, "", 9)
            pdf.set_text_color(100, 100, 100)
            pdf.multi_cell(pdf.epw, 5, safe_text(f"[Attached files: {', '.join(attachments)}]"))

        if text:
            # Strip basic Markdown markers for clean PDF reading
            plain_body = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
            plain_body = re.sub(r"`{1,3}", "", plain_body)
            pdf.set_font(font_family, "", 10)
            pdf.set_text_color(30, 30, 30)
            pdf.multi_cell(pdf.epw, 6, safe_text(plain_body))

        pdf.ln(3)
        y = pdf.get_y()
        pdf.set_draw_color(220, 220, 220)
        pdf.line(pdf.l_margin, y, pdf.w - pdf.r_margin, y)
        pdf.ln(4)

    pdf.output(out_path)

    return out_path

## Generating the Exported File

In [15]:
def generate_export_file(history, export_format):
    """Generates either a Markdown or PDF file whenever the chat or format selection updates."""

    if export_format == "PDF (.pdf)":
        return export_to_pdf(history)

    return export_to_markdown(history)

# **3. DeepSeek ASR (Speech-to-Text) & TTS (Text-to-Speech) Helpers**

## Audio to 16 kHz Mono 16-bit PCM WAV

In [16]:
def convert_to_16bit_pcm_wav(input_path):
    """Converts browser/uploaded audio into 16kHz mono 16-bit PCM WAV required by DeepSeek ASR."""

    temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
    temp_wav.close()

    try:
        subprocess.run(
            ["ffmpeg", "-y", "-i", input_path, "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", temp_wav.name],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

        return temp_wav.name

    except Exception:
        if os.path.exists(temp_wav.name):
            os.remove(temp_wav.name)

        return input_path

## Audio Transcript Generation

In [17]:
def transcribe_audio_with_deepseek(audio_path):
    """Sends audio to DeepSeek's /v1/tools/asr endpoint and returns the transcript."""

    wav_path = convert_to_16bit_pcm_wav(audio_path)

    try:
        with open(wav_path, "rb") as f:
            resp = requests.post(
                f"{DEEPSEEK_BASE_URL}/v1/tools/asr",
                headers={"Authorization": f"Bearer {DEEPSEEK_API_KEY}"},
                files={"file": ("speech.wav", f, "audio/wav")},
                timeout=60
            )

        resp.raise_for_status()

        return resp.json().get("text", "").strip()

    finally:
        if wav_path != audio_path and os.path.exists(wav_path):
            os.remove(wav_path)

## Synthesizing Spoken WAV Audio

In [18]:
def synthesize_speech_with_deepseek(text, voice_id="tide"):
    """Synthesizes spoken WAV audio using DeepSeek's /v1/tools/tts endpoint."""

    clean_text = text.strip()

    if not clean_text:
        return None

    payload = {
        "text": clean_text[:10000],
        "voice_id": voice_id,
        "format": "wav",
        "filter_markdown": True
    }

    resp = requests.post(
        f"{DEEPSEEK_BASE_URL}/v1/tools/tts",
        headers={
            "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
            "Content-Type": "application/json"
        },
        json=payload,
        timeout=90
    )

    resp.raise_for_status()

    out_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
    out_wav.write(resp.content)
    out_wav.close()

    return out_wav.name

# **4. Document & Image Helpers**

## Extracting Text from Document

In [19]:
def extract_document_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    filename = os.path.basename(file_path)

    try:
        if ext == ".pdf":
            reader = PdfReader(file_path)
            text = "\n".join([page.extract_text() or "" for page in reader.pages])

        elif ext == ".docx":
            doc = docx.Document(file_path)
            text = "\n".join([para.text for para in doc.paragraphs])

        else:
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()

        return f"\n\n--- Attached File: {filename} ---\n{text}\n--- End of {filename} ---\n"

    except Exception as e:
        return f"\n\n[Could not read file {filename}: {str(e)}]\n"

## Encoding the Image

In [20]:
def encode_image_to_data_url(file_path):
    mime_type, _ = mimetypes.guess_type(file_path)

    if not mime_type:
        mime_type = "image/png"

    with open(file_path, "rb") as img_file:
        b64_data = base64.b64encode(img_file.read()).decode("utf-8")

    return f"data:{mime_type};base64,{b64_data}"

## Building the Content

In [21]:
def build_api_content(text, files, allow_images=True):
    image_files = [f for f in files if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS]

    doc_files = [
        f for f in files
        if os.path.splitext(f)[1].lower() not in IMAGE_EXTENSIONS
        and os.path.splitext(f)[1].lower() not in AUDIO_EXTENSIONS
    ]

    combined_text = text.strip()

    for doc_path in doc_files:
        combined_text += extract_document_text(doc_path)

    if image_files and allow_images:
        prompt_text = combined_text if combined_text else "Please examine and describe the attached image(s)."
        content_blocks = [{"type": "text", "text": prompt_text}]

        for img_path in image_files:
            content_blocks.append({
                "type": "image_url",
                "image_url": {
                    "url": encode_image_to_data_url(img_path),
                    "detail": "high"
                }
            })

        return content_blocks

    return combined_text if combined_text else "Please analyze the attached file(s)."

# **5. Core Chat Function (Voice + Vision + Docs + Web Search + Streaming)**

In [22]:
def chat_with_kumar(user_input, history, model_label, web_search_enabled, tts_enabled, voice_label, system_prompt):
    text = user_input.get("text", "").strip()
    files = user_input.get("files", [])

    if not text and not files:
        yield history, None
        return

    audio_files = [f for f in files if os.path.splitext(f)[1].lower() in AUDIO_EXTENSIONS]
    non_audio_files = [f for f in files if os.path.splitext(f)[1].lower() not in AUDIO_EXTENSIONS]



    transcripts = []

    if audio_files:
        for audio_path in audio_files:
            try:
                spoken_text = transcribe_audio_with_deepseek(audio_path)

                if spoken_text:
                    transcripts.append(spoken_text)

            except Exception as e:
                history = history + [
                    {"role": "user", "content": [{"type": "file", "file": {"path": audio_path}}]},
                    {"role": "assistant", "content": f"⚠️ **Voice Transcription Error:** {str(e)}"}
                ]

                yield history, None
                return

    joined_transcript = " ".join(transcripts).strip()

    if text and joined_transcript:
        final_user_text = f"{text}\n\n[Voice Transcript]: {joined_transcript}"
        display_text = f"{text}\n\n🎙️ *\"{joined_transcript}\"*"

    elif joined_transcript:
        final_user_text = joined_transcript
        display_text = f"🎙️ *\"{joined_transcript}\"*"

    else:
        final_user_text = text
        display_text = text

    selected_model, enable_thinking = MODEL_OPTIONS[model_label]
    has_images = any(os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS for f in non_audio_files)



    auto_switched_reasons = []

    if has_images and selected_model != "deepseek-flash":
        auto_switched_reasons.append("image analysis")

    if web_search_enabled and selected_model != "deepseek-flash":
        auto_switched_reasons.append("live web search")

    if auto_switched_reasons:
        effective_model = "deepseek-flash"
        reason_str = " & ".join(auto_switched_reasons)
        auto_switched_note = f"*(Switched to **DeepSeek-V4.1-Flash** for {reason_str})*\n\n"

    else:
        effective_model = selected_model
        auto_switched_note = ""

    allow_images = (effective_model == "deepseek-flash")

    if files:
        ui_user_content = [{"type": "file", "file": {"path": f}} for f in files]

        if display_text:
            ui_user_content.append({"type": "text", "text": display_text})

    else:
        ui_user_content = display_text

    api_messages = [{"role": "system", "content": system_prompt}]

    for idx, msg in enumerate(history):
        if idx == 0 and msg["role"] == "assistant":
            continue

        role = msg["role"]
        raw_content = msg["content"]

        if isinstance(raw_content, str):
            clean_text = re.sub(r"^\*\(Switched to.*?\)\*\s*", "", raw_content)
            clean_text = re.sub(r"<details.*?</details>\s*", "", clean_text, flags=re.DOTALL).strip()

            if clean_text:
                api_messages.append({"role": role, "content": clean_text})

        elif isinstance(raw_content, list):
            past_files = []
            past_texts = []

            for block in raw_content:
                if isinstance(block, dict):

                    if role == "user" and block.get("type") == "file" and "file" in block:
                        past_files.append(block["file"]["path"])

                    elif block.get("type") == "text":
                        t = re.sub(r"^\*\(Switched to.*?\)\*\s*", "", block.get("text", ""))
                        t = re.sub(r"<details.*?</details>\s*", "", t, flags=re.DOTALL).strip()

                        if t:
                            past_texts.append(t)

            if role == "assistant":
                if past_texts:
                    api_messages.append({"role": "assistant", "content": "\n".join(past_texts)})

            else:
                api_messages.append({
                    "role": role,
                    "content": build_api_content("\n".join(past_texts), past_files, allow_images=allow_images)
                })

    api_messages.append({
        "role": "user",
        "content": build_api_content(final_user_text, non_audio_files, allow_images=allow_images)
    })

    history = history + [
        {"role": "user", "content": ui_user_content},
        {"role": "assistant", "content": ""}
    ]

    yield history, None

    request_extra_body = {
        "thinking": {"type": "enabled" if enable_thinking else "disabled"}
    }

    if web_search_enabled:
        request_extra_body["tools"] = [
            {"type": "deepseek_web_search"},
            {"type": "deepseek_read_url"}
        ]

    try:
        response = client.chat.completions.create(
            model=effective_model,
            messages=api_messages,
            stream=True,
            extra_body=request_extra_body
        )

        reasoning_text = ""
        answer_text = ""

        for chunk in response:
            delta = chunk.choices[0].delta

            if hasattr(delta, "reasoning_content") and delta.reasoning_content:
                reasoning_text += delta.reasoning_content

            if delta.content:
                answer_text += delta.content

            if reasoning_text:
                thought_title = "🧠🔍 <b>Kumar's Deep Search & Thought Process</b>" if web_search_enabled else "🧠 <b>Kumar's Thought Process</b>"
                formatted_output = (
                    f"{auto_switched_note}"
                    f"<details open><summary>{thought_title}</summary>\n\n"
                    f"{reasoning_text}\n\n</details>\n\n{answer_text}"
                )

            else:
                formatted_output = f"{auto_switched_note}{answer_text}"

            history[-1]["content"] = formatted_output

            yield history, None

        if tts_enabled and answer_text.strip():
            history[-1]["content"] = f"{formatted_output}\n\n*🔊 Synthesizing Kumar's voice...*"
            yield history, None

            try:
                voice_id = VOICE_OPTIONS[voice_label]
                tts_wav_path = synthesize_speech_with_deepseek(answer_text, voice_id=voice_id)

                if tts_wav_path:
                    history[-1]["content"] = [
                        {"type": "text", "text": formatted_output},
                        {"type": "file", "file": {"path": tts_wav_path}}
                    ]
                    yield history, tts_wav_path

                else:
                    history[-1]["content"] = formatted_output
                    yield history, None

            except Exception as tts_err:
                history[-1]["content"] = f"{formatted_output}\n\n*(⚠️ Voice playback error: {str(tts_err)})*"
                yield history, None

    except Exception as e:
        history[-1]["content"] = f"⚠️ **API Error:** {str(e)}\n\nPlease check your DeepSeek API key and account balance."
        yield history, None

# **6. Custom CSS for styling and the footer signature**

In [32]:
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Cinzel:wght@700;800&family=Montserrat:wght@500;600&display=swap');

/* Top-Left Justified Brand Header */
.brand-container {
    display: flex;
    align-items: center;
    gap: 12px;
    padding: 4px 0;
}

.brand-icon {
    font-size: 2.8rem;
    line-height: 1;
    user-select: none;
}

.brand-text-stack {
    display: inline-flex;
    flex-direction: column;
    align-items: stretch;
    width: fit-content;
}

.brand-title {
    font-family: 'Cinzel', Georgia, serif;
    font-size: 2.35rem;
    font-weight: 800;
    line-height: 1.0;
    display: flex;
    justify-content: space-between;
    width: 100%;
    letter-spacing: 0.02em;
}

.brand-subtitle {
    font-family: 'Montserrat', sans-serif;
    font-size: 0.74rem;
    font-weight: 600;
    line-height: 1.25;
    margin-top: 3px;
    letter-spacing: 0.06em;
    white-space: nowrap;
}

/* Export Options ("Markdown" and "PDF") Always in Black Text */
#export-format-radio,
#export-format-radio *,
body.flagship-theme #export-format-radio label,
body.flagship-theme #export-format-radio label span,
body.dark #export-format-radio label,
body.dark #export-format-radio label span {
    color: #000000 !important;
}

body.flagship-theme #export-format-radio label,
body.dark #export-format-radio label {
    background: rgba(255, 255, 255, 0.90) !important;
    border: 1px solid rgba(255, 255, 255, 0.95) !important;
}

/* Smallest Grey Footer Signature */
.footer-signature {
    text-align: center;
    margin-top: 10px;
    padding-top: 8px;
    font-size: 0.65rem !important;
    line-height: 1.2;
    color: #808080 !important;
    border-top: 1px solid rgba(128, 128, 128, 0.22);
}

/* =========================================================
   FLAGSHIP THEME: Bottom-Left Navy Blue -> Top-Right Deep Orange
   with Gentle Bidirectional Water-Ripple Animation & White Text
   ========================================================= */
body.flagship-theme {
    background: linear-gradient(to top right, #00142e 0%, #082a54 42%, #c44700 78%, #f05a00 100%) fixed !important;
    color: #ffffff !important;
    --body-background-fill: transparent;
    --background-fill-primary: rgba(0, 14, 34, 0.42);
    --background-fill-secondary: rgba(0, 14, 34, 0.30);
    --block-background-fill: rgba(0, 14, 34, 0.38);
    --block-border-color: rgba(255, 255, 255, 0.25);
    --block-label-background-fill: rgba(0, 14, 34, 0.55);
    --block-label-text-color: #ffffff;
    --block-title-text-color: #ffffff;
    --body-text-color: #ffffff;
    --body-text-color-subdued: #f0f4f8;
    --code-background-fill: rgba(0, 10, 26, 0.78);
    --input-background-fill: rgba(0, 14, 34, 0.48);
    --input-border-color: rgba(255, 255, 255, 0.32);
    --input-text-color: #ffffff;
    --input-placeholder-color: rgba(255, 255, 255, 0.72);
    --checkbox-background-color: rgba(0, 14, 34, 0.55);
    --checkbox-border-color: rgba(255, 255, 255, 0.45);
    --checkbox-label-text-color: #ffffff;
    --button-secondary-background-fill: rgba(255, 255, 255, 0.14);
    --button-secondary-border-color: rgba(255, 255, 255, 0.32);
    --button-secondary-text-color: #ffffff;
    --button-primary-background-fill: rgba(240, 90, 0, 0.85);
    --button-primary-text-color: #ffffff;
}

/* Gentle water-ripple wave layer:
   0s -> 5s sweeps bottom-left to top-right; 5s -> 10s reverses top-right to bottom-left */
body.flagship-theme::before {
    content: "";
    position: fixed;
    inset: 0;
    pointer-events: none;
    z-index: 0;
    background-image:
        repeating-linear-gradient(
            45deg,
            rgba(255, 255, 255, 0.00) 0px,
            rgba(255, 255, 255, 0.00) 54px,
            rgba(255, 255, 255, 0.045) 82px,
            rgba(180, 220, 255, 0.03) 104px,
            rgba(255, 255, 255, 0.00) 136px
        ),
        radial-gradient(
            circle at 22% 78%,
            rgba(255, 255, 255, 0.055) 0%,
            rgba(255, 255, 255, 0.00) 34%,
            rgba(255, 255, 255, 0.035) 52%,
            rgba(255, 255, 255, 0.00) 72%
        );
    background-size: 240% 240%, 200% 200%;
    background-position: 0% 100%, 0% 100%;
    animation: flagshipWaterRipple 5s ease-in-out infinite alternate;
}

@keyframes flagshipWaterRipple {
    0% {
        background-position: 0% 100%, 0% 100%;
        opacity: 0.72;
    }
    50% {
        opacity: 1;
    }
    100% {
        background-position: 100% 0%, 100% 0%;
        opacity: 0.72;
    }
}

body.flagship-theme .gradio-container {
    position: relative;
    z-index: 1;
    background: transparent !important;
}

body.flagship-theme .gradio-container *:not(.footer-signature):not(#export-format-radio):not(#export-format-radio *) {
    color: #ffffff !important;
}

body.flagship-theme input::placeholder,
body.flagship-theme textarea::placeholder {
    color: rgba(255, 255, 255, 0.75) !important;
}

/* Fix Code / Formula Blocks & Copy Icons in Flagship Theme */
body.flagship-theme pre,
body.flagship-theme .code_wrap,
body.flagship-theme .message pre {
    background: rgba(0, 10, 26, 0.78) !important;
    border: 1px solid rgba(255, 255, 255, 0.25) !important;
    border-radius: 8px !important;
    color: #ffffff !important;
}

body.flagship-theme pre code,
body.flagship-theme pre code * {
    background: transparent !important;
    color: #ffffff !important;
}

body.flagship-theme :not(pre) > code {
    background: rgba(0, 10, 26, 0.68) !important;
    border: 1px solid rgba(255, 255, 255, 0.22) !important;
    color: #ffffff !important;
    padding: 2px 6px !important;
    border-radius: 4px !important;
}

body.flagship-theme pre button,
body.flagship-theme .code_wrap button,
body.flagship-theme .copy_code_button,
body.flagship-theme button[title*="Copy"],
body.flagship-theme button[aria-label*="Copy"] {
    background: rgba(255, 255, 255, 0.14) !important;
    border: 1px solid rgba(255, 255, 255, 0.30) !important;
    color: #ffffff !important;
    opacity: 1 !important;
}

body.flagship-theme pre button:hover,
body.flagship-theme .code_wrap button:hover,
body.flagship-theme .copy_code_button:hover,
body.flagship-theme button[title*="Copy"]:hover,
body.flagship-theme button[aria-label*="Copy"]:hover {
    background: rgba(240, 90, 0, 0.75) !important;
}

body.flagship-theme pre button svg,
body.flagship-theme .code_wrap button svg,
body.flagship-theme .copy_code_button svg,
body.flagship-theme button[title*="Copy"] svg,
body.flagship-theme button[aria-label*="Copy"] svg {
    color: #ffffff !important;
}

/* Keep dropdown option menus readable in Flagship Theme */
body.flagship-theme ul.options,
body.flagship-theme [role="listbox"] {
    background: #061c3a !important;
    border: 1px solid rgba(255, 255, 255, 0.3) !important;
}

body.flagship-theme ul.options li:hover,
body.flagship-theme [role="option"]:hover {
    background: #d94e00 !important;
}

/* Chat bubbles & panels in Flagship Theme */
body.flagship-theme .message.bot,
body.flagship-theme .message.user {
    background: rgba(0, 14, 34, 0.48) !important;
    border: 1px solid rgba(255, 255, 255, 0.22) !important;
    backdrop-filter: blur(8px);
}

body.flagship-theme .footer-signature {
    color: #b0b6bf !important;
    border-top-color: rgba(255, 255, 255, 0.22);
}
"""

# **7. Build the Gradio 6.0 Blocks Interface**

In [30]:
with gr.Blocks(title="Kumar - The Indian AI Companion") as demo:

    # Top Bar: Justified Brand Header, Model Switcher, Voice, Toggles, and 3-Mode Theme Selector
    with gr.Row(equal_height=True):
        with gr.Column(scale=3):
            gr.HTML("""
                <div class="brand-container">
                    <div class="brand-icon">🤖</div>
                    <div class="brand-text-stack">
                        <div class="brand-title">
                            <span>K</span><span>U</span><span>M</span><span>A</span><span>R</span>
                        </div>
                        <div class="brand-subtitle">THE INDIAN AI COMPANION</div>
                    </div>
                </div>
            """)
        with gr.Column(scale=3):
            model_selector = gr.Dropdown(
                choices=list(MODEL_OPTIONS.keys()),
                value="DeepSeek-V4.1-Flash (Fast + Vision + Web)",
                label="Select DeepSeek Model",
                interactive=True
            )
        with gr.Column(scale=2):
            voice_selector = gr.Dropdown(
                choices=list(VOICE_OPTIONS.keys()),
                value="Tide (Male — Kumar's Voice)",
                label="Kumar's Voice",
                interactive=True
            )
        with gr.Column(scale=2):
            web_search_toggle = gr.Checkbox(
                value=True,
                label="🌐 Web Search & URL Reader"
            )
            tts_toggle = gr.Checkbox(
                value=True,
                label="🔊 Spoken Replies (TTS)"
            )
        with gr.Column(scale=2):
            theme_selector = gr.Dropdown(
                choices=["Light Theme", "Dark Theme", "Flagship Theme"],
                value="Flagship Theme",
                label="🎨 Interface Theme",
                interactive=True
            )

    # Optional collapsible system prompt customization
    with gr.Accordion("⚙️ Customize Kumar's Persona (Optional)", open=False):
        system_prompt_input = gr.Textbox(
            value="You are Kumar, a warm, empathetic, and superintelligent AI assistant created by Subhadeep Dey. Keep spoken replies natural, clear, and conversational.",
            label="System Instructions",
            lines=2
        )

    # Main Chatbot Window pre-loaded with Kumar's warm greeting
    chatbot = gr.Chatbot(
        value=[{"role": "assistant", "content": INITIAL_GREETING}],
        height=460
    )

    # Autoplay Voice Output Bar for Hands-Free Replies
    voice_output = gr.Audio(
        label="🔊 Kumar's Live Voice Output",
        autoplay=True,
        interactive=False
    )

    # Multimodal Input Box (Microphone + File Uploads + Text)
    chat_input = gr.MultimodalTextbox(
        sources=["upload", "microphone"],
        interactive=True,
        file_count="multiple",
        placeholder="Type a message, click the microphone to speak, or attach images, audio & documents...",
        show_label=False
    )

    # Action Toolbar: Reset Conversation + Export as Markdown or PDF
    with gr.Row(equal_height=True):
        with gr.Column(scale=2):
            clear_btn = gr.Button("🗑️ Reset Conversation", size="sm")
        with gr.Column(scale=2):
            export_format = gr.Radio(
                choices=[("Markdown", "Markdown (.md)"), ("PDF", "PDF (.pdf)")],
                value="Markdown (.md)",
                label="Export Format",
                container=False,
                elem_id="export-format-radio"
            )
        with gr.Column(scale=2):
            export_btn = gr.DownloadButton(
                label="📥 Export Conversation",
                value=generate_export_file,
                inputs=[chatbot, export_format],
                size="sm",
                variant="secondary"
            )

    # Smallest Grey Footer Signature
    gr.HTML(
        "<div class='footer-signature'>This superintelligent chatbot is built by Subhadeep Dey, copyright 2026</div>"
    )

    # Apply Flagship Theme automatically on initial page load
    demo.load(
        fn=None,
        inputs=None,
        outputs=None,
        js="""() => {
            document.body.classList.remove('dark');
            document.body.classList.add('flagship-theme');
        }"""
    )

    # Event Wiring: Submitting voice, text, URLs, or files
    chat_input.submit(
        fn=chat_with_kumar,
        inputs=[chat_input, chatbot, model_selector, web_search_toggle, tts_toggle, voice_selector, system_prompt_input],
        outputs=[chatbot, voice_output]
    ).then(
        fn=lambda: gr.MultimodalTextbox(value=None),
        inputs=None,
        outputs=[chat_input]
    )

    # Event Wiring: Resetting the chat keeps Kumar's initial greeting
    clear_btn.click(
        fn=lambda: ([{"role": "assistant", "content": INITIAL_GREETING}], None),
        inputs=None,
        outputs=[chatbot, voice_output]
    )

    # Event Wiring: Instant client-side 3-theme switcher (Light / Dark / Flagship)
    theme_selector.change(
        fn=None,
        inputs=[theme_selector],
        outputs=None,
        js="""(theme) => {
            document.body.classList.remove('dark', 'flagship-theme');
            if (theme === 'Dark Theme') {
                document.body.classList.add('dark');
            } else if (theme === 'Flagship Theme') {
                document.body.classList.add('flagship-theme');
            }
        }"""
    )

demo.launch(share=True, debug=True, css=custom_css)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://29ebb4c32bb3ad8817.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3318b0cb9525cff4c9.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://29ebb4c32bb3ad8817.gradio.live
